# Reproduction checks for van der Drift & O'Gorman (2025)
Two checks to run **on the HPC** where the model output lives. Run the first two cells, then either section.

* **Check A, sampling robustness**: is our large dynamic term (0.63 %/% vs the paper's 0.23) just noise?
* **Check B, bin3D reader**: does our pure-Python `.bin3D` reader agree with SAM's own `bin3D2nc`?

In [ ]:
# RUN THIS CELL FIRST: lets this notebook find the samheat code in this repo (no install needed)
import os, sys, pathlib
def _find_repo():
    starts = [pathlib.Path(os.environ.get("JPY_SESSION_NAME", ".")).resolve(), pathlib.Path.cwd().resolve()]
    for s in starts:
        for d in [s, *s.parents]:
            if (d / "samheat" / "__init__.py").exists():
                return d
    for d in [pathlib.Path("/scratch/ttg2015/Torch/ExtremeHeat/SAM_HEAT"), pathlib.Path.home() / "SAM_HEAT"]:
        if (d / "samheat" / "__init__.py").exists():
            return d
    raise FileNotFoundError("Could not find the SAM_HEAT folder. Set REPO by hand, e.g. REPO = pathlib.Path('/scratch/.../SAM_HEAT')")
REPO = _find_repo()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
import samheat
print("using samheat from", pathlib.Path(samheat.__file__).parent)

In [ ]:
# ---- CONFIG (the only cell the others depend on) ----
from samheat.paths import SITE
EXPERIMENT = SITE.work_root / 'vdd2025_repro'   # same name as EXPERIMENT in 01_Launch_Control
RV_LIST = [0, 200, 500, 1000, 2000]     # evaporative resistances r_v (s/m)
LAST_DAYS = 30                          # analysis period: last 30 days
Q = 99.9                                # extreme percentile
BLOCK = 8                               # bootstrap block: 8 x 8 gridpoints x 1 snapshot
N_BOOT = 100                            # bootstrap resamples

## Check A: sub-period robustness
The analysis period (last `LAST_DAYS` days) is split into 3 equal consecutive sub-periods (e.g. days 30-40, 40-50, 50-60). For each sub-period and r_v we compute P_e, C_e and the extreme profiles, then run the decomposition across the r_v series. Results are cached in `<EXPERIMENT>/analysis_cache/subperiods_*.pkl` (delete to recompute).

In [ ]:
from samheat.analysis import repro
N_SUB = 3
df, per_run = repro.subperiod_summaries(EXPERIMENT, RV_LIST, n_sub=N_SUB, last_days=LAST_DAYS, q=Q)
tab = df[df.scope == 'total'].pivot(index='period', columns='quantity', values='value')[['P', 'eff', 'C', 'dynamic', 'thermo', 'residual']]
display(tab.round(3))
display(df[df.scope != 'total'].pivot_table(index=['quantity', 'scope'], columns='period', values='value').round(3))
display(per_run)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from samheat.analysis import repro
df, per_run = repro.subperiod_summaries(EXPERIMENT, RV_LIST, n_sub=3, last_days=LAST_DAYS, q=Q)
full = repro.load_summaries(EXPERIMENT, RV_LIST, last_days=LAST_DAYS, q=Q, block=BLOCK, n_boot=N_BOOT)
ft = repro.decompose(full)['total']
paper = dict(P=1.1, eff=0.8, C=0.3, dynamic=0.23, thermo=0.06)
names = list(paper)
tot = df[df.scope == 'total'].pivot(index='period', columns='quantity', values='value')
fig, ax = plt.subplots(figsize=(8, 4))
for i, (lab, row) in enumerate(tot.iterrows()):
    ax.plot(range(len(names)), [row[n] for n in names], 'o', color='C0', alpha=.6, label='sub-periods' if i == 0 else None)
ax.plot(range(len(names)), [float(ft[n]) for n in names], 's', color='k', ms=9, mfc='none', label='full period')
ax.plot(range(len(names)), [paper[n] for n in names], '*', color='C3', ms=12, label='paper')
ax.set_xticks(range(len(names))); ax.set_xticklabels(names)
ax.set_ylabel('scaling rate (% per %)'); ax.legend(); ax.grid(alpha=.3)
plt.show()
print('dynamic: sub-periods', tot['dynamic'].round(3).tolist(), '| full', round(float(ft['dynamic']), 3), '| paper 0.23')

**How to read it.** If the sub-period dynamic values scatter by about 0.4 %/% (or the sub-period range comfortably spans the paper's 0.23), the discrepancy may be sampling noise. If all sub-periods sit near 0.6, it is systematic (model setup, definition of the extremes, or the decomposition), not noise. The same applies to the other terms; compare the per-pair table for which r_v step drives the scatter.

## Check B: bin3D reader vs SAM's `bin3D2nc`
Copies ONE `.bin3D` from `stage2_rv0/OUT_3D` to a temp directory, converts the copy with SAM's UTIL `bin3D2nc` (modules from `SITE`), and compares every common variable and the coordinates with `read_bin3D`. Tolerance 1e-5 relative (bin3D stores float32). Also shows whether W is on the same vertical levels in both (see the `z` row and W's `shape_ok`).

In [ ]:
from samheat.io.sam3d import find_3d
from samheat.io.check3d import compare_bin3d_with_converter
from samheat.analysis.repro import run_dir
files = find_3d(run_dir(EXPERIMENT, RV_LIST[0]))
print(len(files), 'bin3D files; checking', files[len(files) // 2].name)
res = compare_bin3d_with_converter(files[len(files) // 2], site=SITE)
display(res)